In [ ]:
# In[1]:

import pandas as pd
import pytz

# reuse variable names across runs as kernel is stateful
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames and keep them in required kernel variables
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Helper to convert unix-second timestamp to timezone-aware datetime (UTC+8)
def to_local_ts(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# Add a timezone-aware timestamp column for each dataframe for summary use
metric_df['ts'] = to_local_ts(metric_df['timestamp'])
trace_df['ts'] = to_local_ts(trace_df['timestamp'])
log_df['ts'] = to_local_ts(log_df['timestamp'])
error_logs_df['ts'] = to_local_ts(error_logs_df['timestamp'])

# Compact summary builder
def build_summary(df, name_col=None, message_mode=False):
    total_rows = int(df.shape[0])
    min_ts = df['ts'].min()
    max_ts = df['ts'].max()
    unique_cmdb = list(pd.Series(df['cmdb_id'].dropna().unique())[:50])
    if message_mode:
        # For error_logs: return up to 20 distinct message prefixes or values
        if 'message' in df.columns:
            # take first 200 chars as a "prefix" for uniqueness and readability
            msgs = df['message'].fillna('').astype(str).map(lambda x: x[:200])
            unique_msgs = list(pd.Series(msgs.unique())[:20])
        else:
            unique_msgs = []
        names = unique_msgs
    else:
        if name_col and name_col in df.columns:
            names = list(pd.Series(df[name_col].dropna().unique())[:50])
        else:
            names = []
    # prepare a one-row summary DataFrame (compact)
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'min_ts': min_ts,
        'max_ts': max_ts,
        'unique_cmdb_ids_sample (<=50)': unique_cmdb,
        'unique_names_sample (<=50 or <=20 for error_logs)': names
    }])
    return summary

# Build summaries
summary_metric = build_summary(metric_df, name_col='kpi_name', message_mode=False)
summary_trace = build_summary(trace_df, name_col='trace_name', message_mode=False)
summary_log = build_summary(log_df, name_col='log_name', message_mode=False)
summary_error = build_summary(error_logs_df, message_mode=True)

# Keep compact samples (first 10 rows) for each file
metric_sample = metric_df.head(10)
trace_sample = trace_df.head(10)
log_sample = log_df.head(10)
error_logs_sample = error_logs_df.head(10)

# Final displayed outputs (compact). The full DataFrames are stored in kernel variables.
summary_metric, metric_sample, summary_trace, trace_sample, summary_log, log_sample, summary_error, error_logs_sample

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files (times shown in UTC+8):\n\n"
    "metric.csv: 321,720 rows spanning 2022-03-21 23:00 to 2022-03-21 23:59. "
    "Sample cmdb_ids include adservice, adservice-0, adservice-1, ... . "
    "Sample KPI names include app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, ... . "
    "First 10-row sample was displayed.\n\n"
    "trace.csv: 22,408 rows spanning 2022-03-21 23:30 to 2022-03-21 23:59. "
    "Sample cmdb_ids include adservice-0, adservice-1, adservice-2, ... . "
    "Sample trace names include trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, ... . "
    "First 10-row sample was displayed.\n\n"
    "log.csv: 1,554 rows spanning 2022-03-21 23:30 to 2022-03-21 23:59. "
    "Sample cmdb_ids include adservice-0, adservice-1, adservice-2, cartservice-0, ... . "
    "Log feature names shown include log.error_count and log.row_count. "
    "First 10-row sample was displayed.\n\n"
    "error_logs.csv: 0 rows (empty). No error messages present in the loaded file."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                    min_ts                    max_ts                      unique_cmdb_ids_sample (<=50)  unique_names_sample (<=50 or <=20 for error_logs)
0      321720 2022-03-21 23:00:00+08:00 2022-03-21 23:59:00+08:00  [adservice, adservice-0, adservice-1, adservic...  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,     timestamp    cmdb_id                                           kpi_name        value                        ts
0  1647874800  adservice                                     app.grpc.count    57.000000 2022-03-21 23:00:00+08:00
1  1647874800  adservice                                       app.grpc.mrt     0.002648 2022-03-21 23:00:00+08:00
2  1647874800  adservice                                        app.grpc.rr   100.000000 2022-03-21 23:00:00+08:00
3  1647874800  adservice                                        app.grpc.sr   100.000000 2022-03-21 23:00:00+08:00
4  1647874800  adservice                                     app.http.count    24.000000 2022-03-21 23:00:00+08:00
5  1647874800  adservice                                       app.http.mrt     0.009242 2022-03-21 23:00:00+08:00
6  1647874800  adservice                                        app.http.rr   100.000000 2022-03-21 23:00:00+08:00
7  1647874800  adservice                                        app.http.sr   100.000000 2022-03-21 23:00:00+08:00
8  1647874800  adservice    runtime.java_lang_ClassLoading_LoadedClassCount  5162.000000 2022-03-21 23:00:00+08:00
9  1647874800  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...  5185.000000 2022-03-21 23:00:00+08:00,    total_rows                    min_ts                    max_ts                      unique_cmdb_ids_sample (<=50)  unique_names_sample (<=50 or <=20 for error_logs)
0       22408 2022-03-21 23:30:00+08:00 2022-03-21 23:59:00+08:00  [adservice-0, adservice-1, adservice-2, adserv...  [trace.from_frontend-0.duration_mean, trace.fr...,     timestamp      cmdb_id                           trace_name     value                        ts
0  1647876600  adservice-0  trace.from_frontend-0.duration_mean  0.000018 2022-03-21 23:30:00+08:00
1  1647876600  adservice-0   trace.from_frontend-0.duration_p95  0.000021 2022-03-21 23:30:00+08:00
2  1647876600  adservice-0     trace.from_frontend-0.error_rate  0.000000 2022-03-21 23:30:00+08:00
3  1647876600  adservice-0      trace.from_frontend-0.row_count  6.000000 2022-03-21 23:30:00+08:00
4  1647876600  adservice-0  trace.from_frontend-2.duration_mean  0.000021 2022-03-21 23:30:00+08:00
5  1647876600  adservice-0   trace.from_frontend-2.duration_p95  0.000026 2022-03-21 23:30:00+08:00
6  1647876600  adservice-0     trace.from_frontend-2.error_rate  0.000000 2022-03-21 23:30:00+08:00
7  1647876600  adservice-0      trace.from_frontend-2.row_count  7.000000 2022-03-21 23:30:00+08:00
8  1647876600  adservice-1  trace.from_frontend-0.duration_mean  0.000018 2022-03-21 23:30:00+08:00
9  1647876600  adservice-1   trace.from_frontend-0.duration_p95  0.000023 2022-03-21 23:30:00+08:00,    total_rows                    min_ts                    max_ts                      unique_cmdb_ids_sample (<=50) unique_names_sample (<=50 or <=20 for error_logs)
0        1554 2022-03-21 23:30:00+08:00 2022-03-21 23:59:00+08:00  [adservice-0, adservice-1, adservice-2, cartse...                  [log.error_count, log.row_count],     timestamp        cmdb_id         log_name  value                        ts
0  1647876600    adservice-0  log.error_count    0.0 2022-03-21 23:30:00+08:00
1  1647876600    adservice-0    log.row_count   38.0 2022-03-21 23:30:00+08:00
2  1647876600    adservice-1  log.error_count    0.0 2022-03-21 23:30:00+08:00
3  1647876600    adservice-1    log.row_count   30.0 2022-03-21 23:30:00+08:00
4  1647876600    adservice-2  log.error_count    0.0 2022-03-21 23:30:00+08:00
5  1647876600    adservice-2    log.row_count   26.0 2022-03-21 23:30:00+08:00
6  1647876600  cartservice-0  log.error_count    0.0 2022-03-21 23:30:00+08:00
7  1647876600  cartservice-0    log.row_count  151.0 2022-03-21 23:30:00+08:00
8  1647876600  cartservice-1  log.error_count    0.0 2022-03-21 23:30:00+08:00
9  1647876600  cartservice-1    log.row_count  150.0 2022-03-21 23:30:00+08:00,    total_rows min_ts max_ts unique_cmdb_ids_sample (<=50) unique_names_sample (<=50 or <=20 for error_logs)
0           0    NaT    NaT                            []                                                [], Empty DataFrame
Columns: [timestamp, cmdb_id, message, ts]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2022-03-21 23:30:00', tz=tz)
end   = pd.Timestamp('2022-03-22 00:00:00', tz=tz)  # exclusive

# 1) METRIC ANALYSIS
# a) global P95 per (cmdb_id, kpi_name)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# b) filter to incident window (do not modify original)
metric_window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] < end)].copy()

# c) aggregate anomalies in window joined with global p95
if metric_window_df.shape[0] > 0:
    m_w_merged = metric_window_df.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    def metric_agg_fn(df):
        count_total = int(df.shape[0])
        count_above_p95 = int((df['value'] > df['p95']).sum())
        earliest_anom_ts = df.loc[df['value'] > df['p95'], 'ts'].min() if (df['value'] > df['p95']).any() else pd.NaT
        max_value_in_window = df['value'].max()
        return pd.Series({
            'count_total': count_total,
            'count_above_p95': count_above_p95,
            'earliest_anom_ts': earliest_anom_ts,
            'max_value_in_window': max_value_in_window
        })
    metric_agg = (
        m_w_merged
        .groupby(['cmdb_id','kpi_name','p95'], as_index=False)
        .apply(lambda g: metric_agg_fn(g))
        .reset_index(drop=True)
    )
    metric_agg_sorted = metric_agg.sort_values(['count_above_p95','count_total'], ascending=[False, False])
    metric_agg_result = metric_agg_sorted[['cmdb_id','kpi_name','p95','count_total','count_above_p95','earliest_anom_ts','max_value_in_window']].head(20)
else:
    metric_agg_result = pd.DataFrame({'info': ['<EMPTY> no metric rows in incident window']})

# 2) TRACE ANALYSIS
# a) global P95 and P05 per (cmdb_id, trace_name)
trace_p95 = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95'})
trace_p05 = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value':'p05'})
trace_thresholds = trace_p95.merge(trace_p05, on=['cmdb_id','trace_name'], how='outer')

# b) filter to window
trace_window_df = trace_df[(trace_df['ts'] >= start) & (trace_df['ts'] < end)].copy()

# c) aggregates
if trace_window_df.shape[0] > 0:
    t_w_merged = trace_window_df.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    def trace_agg_fn(df):
        count_total = int(df.shape[0])
        count_above_p95 = int((df['value'] > df['p95']).sum())
        count_below_p05 = int((df['value'] < df['p05']).sum())
        earliest_above_ts = df.loc[df['value'] > df['p95'], 'ts'].min() if (df['value'] > df['p95']).any() else pd.NaT
        earliest_below_ts = df.loc[df['value'] < df['p05'], 'ts'].min() if (df['value'] < df['p05']).any() else pd.NaT
        max_value_in_window = df['value'].max()
        min_value_in_window = df['value'].min()
        return pd.Series({
            'count_total': count_total,
            'count_above_p95': count_above_p95,
            'count_below_p05': count_below_p05,
            'earliest_above_ts': earliest_above_ts,
            'earliest_below_ts': earliest_below_ts,
            'max_value_in_window': max_value_in_window,
            'min_value_in_window': min_value_in_window
        })
    trace_agg = (
        t_w_merged
        .groupby(['cmdb_id','trace_name','p95','p05'], as_index=False)
        .apply(lambda g: trace_agg_fn(g))
        .reset_index(drop=True)
    )
    trace_agg['severity_count'] = trace_agg['count_above_p95'] + trace_agg['count_below_p05']
    trace_agg_sorted = trace_agg.sort_values('severity_count', ascending=False)
    trace_agg_result = trace_agg_sorted[['cmdb_id','trace_name','p95','p05','count_total','count_above_p95','count_below_p05','earliest_above_ts','earliest_below_ts','max_value_in_window','min_value_in_window']].head(20)
else:
    trace_agg_result = pd.DataFrame({'info': ['<EMPTY> no trace rows in incident window']})

# 3) LOG ANALYSIS
# a) global P95 and P05 per (cmdb_id, log_name)
log_p95 = log_df.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95'})
log_p05 = log_df.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value':'p05'})
log_thresholds = log_p95.merge(log_p05, on=['cmdb_id','log_name'], how='outer')

# b) filter to window
log_window_df = log_df[(log_df['ts'] >= start) & (log_df['ts'] < end)].copy()

# c) aggregates
if log_window_df.shape[0] > 0:
    l_w_merged = log_window_df.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    def log_agg_fn(df):
        count_total = int(df.shape[0])
        count_above_p95 = int((df['value'] > df['p95']).sum())
        count_below_p05 = int((df['value'] < df['p05']).sum())
        earliest_above_ts = df.loc[df['value'] > df['p95'], 'ts'].min() if (df['value'] > df['p95']).any() else pd.NaT
        earliest_below_ts = df.loc[df['value'] < df['p05'], 'ts'].min() if (df['value'] < df['p05']).any() else pd.NaT
        max_value_in_window = df['value'].max()
        min_value_in_window = df['value'].min()
        return pd.Series({
            'count_total': count_total,
            'count_above_p95': count_above_p95,
            'count_below_p05': count_below_p05,
            'earliest_above_ts': earliest_above_ts,
            'earliest_below_ts': earliest_below_ts,
            'max_value_in_window': max_value_in_window,
            'min_value_in_window': min_value_in_window
        })
    log_agg = (
        l_w_merged
        .groupby(['cmdb_id','log_name','p95','p05'], as_index=False)
        .apply(lambda g: log_agg_fn(g))
        .reset_index(drop=True)
    )
    log_agg['severity_count'] = log_agg['count_above_p95'] + log_agg['count_below_p05']
    log_agg_sorted = log_agg.sort_values('severity_count', ascending=False)
    log_agg_result = log_agg_sorted[['cmdb_id','log_name','p95','p05','count_total','count_above_p95','count_below_p05','earliest_above_ts','earliest_below_ts','max_value_in_window','min_value_in_window']].head(20)
else:
    log_agg_result = pd.DataFrame({'info': ['<EMPTY> no log rows in incident window']})

# Final returned compact tables and stored full threshold tables:
metric_thresholds, metric_agg_result, trace_thresholds, trace_agg_result, log_thresholds, log_agg_result

```
Out[2]:
```
# concise plain-English summary based on the previous aggregations (UTC+8)
metric_count = metric_thresholds.shape[0] if 'metric_thresholds' in globals() else 0
trace_count = trace_thresholds.shape[0] if 'trace_thresholds' in globals() else 0
log_count = log_thresholds.shape[0] if 'log_thresholds' in globals() else 0

# pick top component examples from aggregated results (safe if empty)
def top_examples(df, col='cmdb_id', n=5):
    if isinstance(df, pd.DataFrame) and not df.empty and col in df.columns:
        return list(pd.Series(df[col].head(n).astype(str)).unique())
    return []

metric_top_comps = top_examples(metric_agg_result, 'cmdb_id', 5)
trace_top_comps = top_examples(trace_agg_result, 'cmdb_id', 5)
log_top_comps = top_examples(log_agg_result, 'cmdb_id', 8)

# earliest anomaly timestamps examples (compact)
def earliest_examples(df, timecol, n=3):
    if isinstance(df, pd.DataFrame) and not df.empty and timecol in df.columns:
        vals = pd.Series(df[timecol].dropna().astype(str).unique())
        return list(vals[:n])
    return []

metric_earlies = earliest_examples(metric_agg_result, 'earliest_anom_ts', 3)
trace_earlies_above = earliest_examples(trace_agg_result, 'earliest_above_ts', 3)
trace_earlies_below = earliest_examples(trace_agg_result, 'earliest_below_ts', 3)
log_earlies_above = earliest_examples(log_agg_result, 'earliest_above_ts', 3)
log_earlies_below = earliest_examples(log_agg_result, 'earliest_below_ts', 3)

summary = (
    "Incident window: 2022-03-21 23:30 to 2022-03-22 00:00 (UTC+8).\n\n"
    f"Thresholds computed: metric groups={metric_count}, trace groups={trace_count}, log groups={log_count}.\n\n"
    "Key observations:\n"
    f"- Metrics: top anomaly groups (examples): {metric_top_comps}. "
    f"Many metric groups (notably for 'adservice') show values exceeding their global P95; "
    f"examples of earliest metric anomaly times: {metric_earlies} (UTC+8). Most metric groups had 30 samples in the window.\n\n"
    f"- Traces: top affected components (examples): {trace_top_comps}. Several trace series (shippingservice and recommendationservice variants) "
    "have both values above P95 and below P05 during the window (counts typically 2 each). "
    f"Earliest above-P95 times: {trace_earlies_above}; earliest below-P05 times: {trace_earlies_below}.\n\n"
    f"- Logs: top components with log.row_count deviations (examples): {log_top_comps}. Many services show both above-P95 and below-P05 log.row_count events (commonly 2 each). "
    f"Earliest above-P95 log times: {log_earlies_above}; earliest below-P05 log times: {log_earlies_below}.\n\n"
    "Summary conclusion: Metric anomalies are concentrated in adservice (multiple JVM/runtime/GC KPIs exceed P95). "
    "Trace and log anomalies are more widespread across shippingservice, recommendationservice, frontend, productcatalogservice, cartservice, and currencyservice — indicating throughput/latency fluctuations across services during the window. "
    "Next step: inspect the listed components' detailed traces and logs around the shown earliest timestamps to determine root cause."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name           p95
0              adservice                                     app.grpc.count     88.050000
1              adservice                                       app.grpc.mrt      0.004163
2              adservice                                        app.grpc.rr    100.000000
3              adservice                                        app.grpc.sr    100.000000
4              adservice                                     app.http.count     24.000000
...                  ...                                                ...           ...
5369  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     21.000000
5370  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  59371.250000
5371  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.002124
5372  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     19.500000
5373  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1892.987500

[5374 rows x 3 columns],        cmdb_id                                           kpi_name           p95  count_total  count_above_p95          earliest_anom_ts  max_value_in_window
0    adservice                                     app.grpc.count  8.805000e+01           30                3 2022-03-21 23:47:00+08:00         9.100000e+01
5    adservice                                       app.http.mrt  1.103865e-02           30                3 2022-03-21 23:34:00+08:00         1.453125e-02
14   adservice  runtime.java_lang_GarbageCollector_CollectionC...  4.102738e+04           30                3 2022-03-21 23:57:00+08:00         4.103475e+04
16   adservice  runtime.java_lang_GarbageCollector_CollectionT...  7.456975e+04           30                3 2022-03-21 23:57:00+08:00         7.458333e+04
22   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  7.338491e+08           30                3 2022-03-21 23:57:00+08:00         7.340279e+08
24   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  4.102738e+04           30                3 2022-03-21 23:57:00+08:00         4.103475e+04
72   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  4.481952e+07           30                3 2022-03-21 23:57:00+08:00         4.481958e+07
120  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  4.481951e+07           30                3 2022-03-21 23:57:00+08:00         4.481958e+07
122  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  7.338491e+08           30                3 2022-03-21 23:57:00+08:00         7.340279e+08
220  adservice  runtime.java_lang_MemoryPool_Usage_used.Tenure...  4.481952e+07           30                3 2022-03-21 23:57:00+08:00         4.481958e+07
244  adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.345259e+12           30                3 2022-03-21 23:57:00+08:00         2.345752e+12
245  adservice    runtime.java_lang_OperatingSystem_SystemCpuLoad  1.439226e-02           30                3 2022-03-21 23:57:00+08:00         1.439239e-02
246  adservice  runtime.java_lang_OperatingSystem_SystemLoadAv...  4.501667e+00           30                3 2022-03-21 23:54:00+08:00         5.160000e+00
251  adservice                   runtime.java_lang_Runtime_Uptime  7.338607e+08           30                3 2022-03-21 23:57:00+08:00         7.340377e+08
252  adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  1.779025e+11           30                3 2022-03-21 23:57:00+08:00         1.779457e+11
253  adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2.459439e+11           30                3 2022-03-21 23:54:00+08:00         2.460763e+11
255  adservice  runtime.java_lang_Threading_CurrentThreadUserTime  2.244343e+11           30                3 2022-03-21 23:54:00+08:00         2.246083e+11
316  adservice        runtime.jvm_memory_pool_MB_used.Tenured_Gen  4.274323e+01           30                3 2022-03-21 23:57:00+08:00         4.274328e+01
325  adservice  runtime.jvm_memory_pool_allocated_MB_total.Ede...  8.707519e+05           30                3 2022-03-21 23:57:00+08:00         8.709618e+05
327  adservice  runtime.jvm_memory_pool_allocated_MB_total.Sur...  2.190273e+03           30                3 2022-03-21 23:57:00+08:00         2.190409e+03,                  cmdb_id                               trace_name        p95       p05
0            adservice-0      trace.from_frontend-0.duration_mean   0.000026  0.000017
1            adservice-0       trace.from_frontend-0.duration_p95   0.000052  0.000019
2            adservice-0         trace.from_frontend-0.error_rate   0.000000  0.000000
3            adservice-0          trace.from_frontend-0.row_count   7.000000  3.000000
4            adservice-0      trace.from_frontend-2.duration_mean   0.000035  0.000017
...                  ...                                      ...        ...       ...
1051  shippingservice2-0  trace.from_checkoutservice2-0.row_count  10.000000  2.000000
1052  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000102  0.000070
1053  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000236  0.000076
1054  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000  0.000000
1055  shippingservice2-0         trace.from_frontend2-0.row_count  25.550000  4.450000

[1056 rows x 4 columns],                        cmdb_id                                      trace_name       p95       p05  count_total  count_above_p95  count_below_p05         earliest_above_ts         earliest_below_ts  max_value_in_window  min_value_in_window
1020         shippingservice-1             trace.from_frontend-0.duration_mean  0.000122  0.000068           26                2                2 2022-03-21 23:45:00+08:00 2022-03-21 23:48:00+08:00             0.000239             0.000066
1021         shippingservice-1              trace.from_frontend-0.duration_p95  0.000170  0.000068           26                2                2 2022-03-21 23:45:00+08:00 2022-03-21 23:48:00+08:00             0.000377             0.000066
1000         shippingservice-0             trace.from_frontend-0.duration_mean  0.000108  0.000067           24                2                2 2022-03-21 23:43:00+08:00 2022-03-21 23:30:00+08:00             0.000170             0.000066
1004         shippingservice-0             trace.from_frontend-2.duration_mean  0.000178  0.000066           24                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:40:00+08:00             0.000236             0.000064
1005         shippingservice-0              trace.from_frontend-2.duration_p95  0.000273  0.000067           24                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:40:00+08:00             0.000373             0.000064
977   recommendationservice2-0   trace.to_productcatalogservice-0.duration_p95  0.000064  0.000031           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:30:00+08:00             0.000093             0.000031
968   recommendationservice2-0            trace.from_frontend2-0.duration_mean  0.002807  0.002685           30                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:37:00+08:00             0.002834             0.002669
976   recommendationservice2-0  trace.to_productcatalogservice-0.duration_mean  0.000043  0.000028           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:30:00+08:00             0.000045             0.000028
944    recommendationservice-2             trace.from_frontend-0.duration_mean  0.002797  0.002491           30                2                2 2022-03-21 23:31:00+08:00 2022-03-21 23:47:00+08:00             0.003185             0.002481
961    recommendationservice-2   trace.to_productcatalogservice-1.duration_p95  0.000042  0.000028           30                2                2 2022-03-21 23:35:00+08:00 2022-03-21 23:30:00+08:00             0.000060             0.000028
973   recommendationservice2-0                         trace.self.duration_p95  0.003062  0.002751           30                2                2 2022-03-21 23:46:00+08:00 2022-03-21 23:34:00+08:00             0.003097             0.002704
972   recommendationservice2-0                        trace.self.duration_mean  0.002425  0.002307           30                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:37:00+08:00             0.002454             0.002291
969   recommendationservice2-0             trace.from_frontend2-0.duration_p95  0.003437  0.003145           30                2                2 2022-03-21 23:46:00+08:00 2022-03-21 23:32:00+08:00             0.003498             0.003074
948    recommendationservice-2             trace.from_frontend-2.duration_mean  0.002791  0.002517           30                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:49:00+08:00             0.002894             0.002490
945    recommendationservice-2              trace.from_frontend-0.duration_p95  0.003454  0.002630           30                2                2 2022-03-21 23:31:00+08:00 2022-03-21 23:46:00+08:00             0.004665             0.002607
949    recommendationservice-2              trace.from_frontend-2.duration_p95  0.003206  0.002621           30                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:52:00+08:00             0.003666             0.002609
965    recommendationservice-2   trace.to_productcatalogservice-2.duration_p95  0.000054  0.000027           30                2                2 2022-03-21 23:45:00+08:00 2022-03-21 23:33:00+08:00             0.000208             0.000027
952    recommendationservice-2                        trace.self.duration_mean  0.002380  0.002162           30                2                2 2022-03-21 23:53:00+08:00 2022-03-21 23:47:00+08:00             0.002519             0.002142
953    recommendationservice-2                         trace.self.duration_p95  0.003042  0.002295           30                2                2 2022-03-21 23:53:00+08:00 2022-03-21 23:47:00+08:00             0.003882             0.002275
925    recommendationservice-1              trace.from_frontend-2.duration_p95  0.003156  0.002579           30                2                2 2022-03-21 23:33:00+08:00 2022-03-21 23:37:00+08:00             0.003217             0.002549,               cmdb_id         log_name    p95    p05
0         adservice-0  log.error_count   0.00   0.00
1         adservice-0    log.row_count  38.00  26.00
2         adservice-1  log.error_count   0.00   0.00
3         adservice-1    log.row_count  27.10  14.90
4         adservice-2  log.error_count   0.00   0.00
..                ...              ...    ...    ...
57  shippingservice-0    log.row_count  34.20   6.80
58  shippingservice-1  log.error_count   0.00   0.00
59  shippingservice-1    log.row_count  33.55   9.45
60  shippingservice-2  log.error_count   0.00   0.00
61  shippingservice-2    log.row_count  31.30   7.80

[62 rows x 4 columns],                     cmdb_id       log_name     p95     p05  count_total  count_above_p95  count_below_p05         earliest_above_ts         earliest_below_ts  max_value_in_window  min_value_in_window
3               adservice-1  log.row_count   27.10   14.90           30                2                2 2022-03-21 23:30:00+08:00 2022-03-21 23:31:00+08:00                 30.0                 12.0
7             cartservice-0  log.row_count  195.85  134.50           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:38:00+08:00                201.0                129.0
11            cartservice-2  log.row_count  203.05  135.95           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:38:00+08:00                228.0                123.0
23        currencyservice-2  log.row_count  180.95  111.05           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:55:00+08:00                195.0                 96.0
19        currencyservice-0  log.row_count  178.80  114.35           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:34:00+08:00                187.0                 94.0
21        currencyservice-1  log.row_count  181.60  110.70           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:43:00+08:00                199.0                101.0
47  productcatalogservice-2  log.row_count  252.30  191.15           30                2                2 2022-03-21 23:41:00+08:00 2022-03-21 23:38:00+08:00                261.0                187.0
51  recommendationservice-1  log.row_count   54.75   37.35           30                2                2 2022-03-21 23:39:00+08:00 2022-03-21 23:38:00+08:00                 58.0                 35.0
53  recommendationservice-2  log.row_count   53.10   39.45           30                2                2 2022-03-21 23:41:00+08:00 2022-03-21 23:52:00+08:00                 55.0                 37.0
57        shippingservice-0  log.row_count   34.20    6.80           29                2                2 2022-03-21 23:33:00+08:00 2022-03-21 23:36:00+08:00                 36.0                  5.0
45  productcatalogservice-1  log.row_count  254.10  189.45           30                2                2 2022-03-21 23:49:00+08:00 2022-03-21 23:38:00+08:00                260.0                185.0
43  productcatalogservice-0  log.row_count  254.20  186.90           30                2                2 2022-03-21 23:49:00+08:00 2022-03-21 23:38:00+08:00                261.0                184.0
35               frontend-2  log.row_count  493.65  323.40           30                2                2 2022-03-21 23:33:00+08:00 2022-03-21 23:57:00+08:00                557.0                299.0
31               frontend-0  log.row_count  478.20  334.25           30                2                2 2022-03-21 23:41:00+08:00 2022-03-21 23:32:00+08:00                496.0                300.0
59        shippingservice-1  log.row_count   33.55    9.45           30                2                2 2022-03-21 23:31:00+08:00 2022-03-21 23:30:00+08:00                 34.0                  6.0
61        shippingservice-2  log.row_count   31.30    7.80           30                2                2 2022-03-21 23:31:00+08:00 2022-03-21 23:32:00+08:00                 35.0                  6.0
9             cartservice-1  log.row_count  190.00  144.05           30                1                2 2022-03-21 23:49:00+08:00 2022-03-21 23:42:00+08:00                209.0                140.0
49  recommendationservice-0  log.row_count   53.00   38.90           30                1                2 2022-03-21 23:41:00+08:00 2022-03-21 23:38:00+08:00                 54.0                 38.0
39         paymentservice-1  log.row_count    6.30    1.90           19                1                1 2022-03-21 23:54:00+08:00 2022-03-21 23:42:00+08:00                  9.0                  1.0
15        checkoutservice-1  log.row_count   19.00   11.00           21                1                1 2022-03-21 23:44:00+08:00 2022-03-21 23:41:00+08:00                 34.0                  3.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Ensure required variables exist in kernel
if 'metric_thresholds' not in globals() or 'metric_window_df' not in globals():
    raise RuntimeError("Required variables metric_thresholds and metric_window_df not found in kernel.")

# Merge p95 into a working copy (do not modify original)
m_w_merged = metric_window_df.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# If no rows or no p95 available, return empty indicators
if m_w_merged.shape[0] == 0 or m_w_merged['p95'].isna().all():
    metric_fault_runs = pd.DataFrame()  # store empty full result
    metric_fault_runs_top20 = pd.DataFrame({'info': ['<EMPTY> no metric rows or no thresholds available in window']})
    runs_per_cmdb_top10 = pd.DataFrame({'info': ['<EMPTY> no metric fault runs']})
else:
    # Sort for consecutive detection
    m = m_w_merged.sort_values(['cmdb_id', 'kpi_name', 'ts']).copy()
    # Boolean flag where value > global p95
    m['above'] = m['value'] > m['p95']
    # Prev above using groupby shift to align indices correctly
    m['prev_above'] = m.groupby(['cmdb_id', 'kpi_name'])['above'].shift(1).fillna(False)
    # Flag change where above status differs from previous row in the series
    m['flag_change'] = (m['above'] != m['prev_above']).astype(int)
    # Run id as cumulative sum of flag_change within each series (ensures aligned index)
    m['run_id'] = m.groupby(['cmdb_id', 'kpi_name'])['flag_change'].cumsum()
    # Keep only anomalous rows (where above is True)
    anom = m[m['above']].copy()
    if anom.shape[0] == 0:
        metric_fault_runs = pd.DataFrame()
        metric_fault_runs_top20 = pd.DataFrame({'info': ['<EMPTY> no anomalous runs (value > p95) in window']})
        runs_per_cmdb_top10 = pd.DataFrame({'info': ['<EMPTY> no anomalous runs']})
    else:
        # Aggregate consecutive anomalous runs per (cmdb_id, kpi_name, run_id)
        agg = (
            anom
            .groupby(['cmdb_id', 'kpi_name', 'run_id'], as_index=False)
            .agg(
                run_length=('value', 'size'),
                start_ts=('ts', 'min'),
                end_ts=('ts', 'max'),
                max_value_in_run=('value', 'max'),
                p95=('p95', 'first')
            )
        )
        # Keep runs with length >= 2
        agg = agg[agg['run_length'] >= 2].copy()
        # Compute breach_ratio, handle p95==0 or NaN
        def compute_breach(p95, maxv):
            if pd.isna(p95) or p95 == 0:
                return np.nan
            return (maxv - p95) / p95
        agg['breach_ratio'] = agg.apply(lambda r: compute_breach(r['p95'], r['max_value_in_run']), axis=1)
        # Store full result in kernel variable
        metric_fault_runs = agg.sort_values('breach_ratio', ascending=False).reset_index(drop=True)
        # Compact top 20 table sorted by breach_ratio desc
        metric_fault_runs_top20 = metric_fault_runs.sort_values('breach_ratio', ascending=False)[
            ['cmdb_id', 'kpi_name', 'p95', 'run_length', 'start_ts', 'end_ts', 'max_value_in_run', 'breach_ratio']
        ].head(20).reset_index(drop=True)
        # Summary: number_of_runs per cmdb_id (top 10)
        runs_per_cmdb = (
            metric_fault_runs
            .groupby('cmdb_id', as_index=False)
            .size()
            .rename(columns={'size': 'number_of_runs'})
            .sort_values('number_of_runs', ascending=False)
        )
        runs_per_cmdb_top10 = runs_per_cmdb.head(10).reset_index(drop=True)

# Return/store compact outputs; full metric_fault_runs is stored in kernel
metric_fault_runs, metric_fault_runs_top20, runs_per_cmdb_top10

```
Out[3]:
```
# Build a concise plain-English summary string using the stored results in the kernel
if 'metric_fault_runs' not in globals() or (isinstance(metric_fault_runs, pd.DataFrame) and metric_fault_runs.empty):
    summary = "No metric fault runs found (no consecutive runs with value > p95 and length >= 2) in the incident window."
else:
    total_runs = int(metric_fault_runs.shape[0])
    # Top run by breach_ratio (already sorted descending)
    top = metric_fault_runs.sort_values('breach_ratio', ascending=False).iloc[0]
    top_summary = (
        f"Top run: {top['cmdb_id']} | {top['kpi_name']} | breach_ratio={top['breach_ratio']:.2f} "
        f"(p95={top['p95']:.6g}) | run_length={int(top['run_length'])} | "
        f"start={pd.Timestamp(top['start_ts']).tz_convert('Asia/Shanghai')} | end={pd.Timestamp(top['end_ts']).tz_convert('Asia/Shanghai')} | "
        f"max_value={top['max_value_in_run']:.6g}"
    )
    # Top 3 runs brief list
    top3 = metric_fault_runs.sort_values('breach_ratio', ascending=False).head(3)[['cmdb_id','kpi_name','breach_ratio','run_length']].to_dict('records')
    top3_list = "; ".join([f"{r['cmdb_id']}/{r['kpi_name']} br={r['breach_ratio']:.2f} len={int(r['run_length'])}" for r in top3])
    # Top components by number_of_runs (from runs_per_cmdb_top10 if present)
    if 'runs_per_cmdb_top10' in globals() and isinstance(runs_per_cmdb_top10, pd.DataFrame) and not runs_per_cmdb_top10.empty:
        comps = ", ".join([f"{row['cmdb_id']}({int(row['number_of_runs'])})" for _, row in runs_per_cmdb_top10.head(10).iterrows()])
        comps_summary = f"Top components by number_of_runs: {comps}."
    else:
        comps_summary = "No per-component run counts available."
    summary = (
        f"Detected {total_runs} metric fault runs (consecutive points > global P95 with length>=2) during the incident window.\n"
        f"{top_summary}\n"
        f"Other top runs: {top3_list}.\n"
        f"{comps_summary}\n"
        "Recommendation: investigate the listed KPIs on the named components (start/end times shown) for root-cause (network or container resource spikes indicated)."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id                                           kpi_name  run_id  run_length                  start_ts                    end_ts  max_value_in_run           p95  breach_ratio
0        emailservice-2  container.node-6.container_network_receive_MB....       1           2 2022-03-21 23:41:00+08:00 2022-03-21 23:42:00+08:00      6.007805e-01  2.687402e-02  2.135544e+01
1         cartservice-2  container.node-6.container_cpu_cfs_throttled_s...       1           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      1.740780e-02  8.703900e-04  1.900000e+01
2         cartservice-2  container.node-6.container_cpu_cfs_throttled_p...       1           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      5.000000e-01  2.500000e-02  1.900000e+01
3      paymentservice-0      container.node-6.container_cpu_system_seconds       1           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      1.000000e-02  5.000000e-04  1.900000e+01
4                node-2                              system.net.bytes_rcvd       3           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      6.693152e+06  7.113528e+05  8.409046e+00
..                  ...                                                ...     ...         ...                       ...                       ...               ...           ...           ...
336  shippingservice2-0               container.node-6.container_last_seen       1           3 2022-03-21 23:57:00+08:00 2022-03-21 23:59:00+08:00      1.647878e+09  1.647878e+09  1.074109e-07
337   currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...       1           2 2022-03-21 23:57:00+08:00 2022-03-21 23:58:00+08:00      5.000000e-01  0.000000e+00           NaN
338   currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...       1           2 2022-03-21 23:57:00+08:00 2022-03-21 23:58:00+08:00      2.182313e-02  0.000000e+00           NaN
339   paymentservice2-0  container.node-6.container_cpu_cfs_throttled_p...       1           2 2022-03-21 23:49:00+08:00 2022-03-21 23:50:00+08:00      5.000000e-01  0.000000e+00           NaN
340   paymentservice2-0  container.node-6.container_cpu_cfs_throttled_s...       1           2 2022-03-21 23:49:00+08:00 2022-03-21 23:50:00+08:00      1.090628e-02  0.000000e+00           NaN

[341 rows x 9 columns],                cmdb_id                                           kpi_name            p95  run_length                  start_ts                    end_ts  max_value_in_run  breach_ratio
0       emailservice-2  container.node-6.container_network_receive_MB....       0.026874           2 2022-03-21 23:41:00+08:00 2022-03-21 23:42:00+08:00      6.007805e-01     21.355440
1        cartservice-2  container.node-6.container_cpu_cfs_throttled_s...       0.000870           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      1.740780e-02     19.000000
2        cartservice-2  container.node-6.container_cpu_cfs_throttled_p...       0.025000           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      5.000000e-01     19.000000
3     paymentservice-0      container.node-6.container_cpu_system_seconds       0.000500           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      1.000000e-02     19.000000
4               node-2                              system.net.bytes_rcvd  711352.847500           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      6.693152e+06      8.409046
5     paymentservice-1  container.node-6.container_network_receive_MB....       0.041667           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      3.123484e-01      6.496261
6      emailservice2-0  container.node-6.container_network_receive_MB....       0.048635           2 2022-03-21 23:35:00+08:00 2022-03-21 23:36:00+08:00      3.225021e-01      5.631122
7               node-2                              system.net.bytes_sent  761985.088500           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      5.021184e+06      5.589609
8    checkoutservice-2  container.node-5.container_network_receive_MB....       0.049876           2 2022-03-21 23:42:00+08:00 2022-03-21 23:43:00+08:00      3.154960e-01      5.325609
9               node-2                        system.net.packets_in.count     318.937500           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      1.995930e+03      5.258060
10       cartservice-2  container.node-6.container_network_receive_MB....       0.056188           2 2022-03-21 23:42:00+08:00 2022-03-21 23:43:00+08:00      3.395877e-01      5.043742
11       cartservice-0  container.node-6.container_network_receive_MB....       0.056641           2 2022-03-21 23:39:00+08:00 2022-03-21 23:40:00+08:00      3.315315e-01      4.853197
12              node-4                              system.net.bytes_rcvd  643706.405000           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      3.698073e+06      4.744968
13              node-4                              system.net.bytes_sent  914096.024000           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      5.199190e+06      4.687794
14        redis-cart-0  container.node-6.container_network_receive_MB....       0.061202           2 2022-03-21 23:49:00+08:00 2022-03-21 23:50:00+08:00      3.356667e-01      4.484596
15   checkoutservice-0  container.node-6.container_network_receive_MB....       0.062355           2 2022-03-21 23:33:00+08:00 2022-03-21 23:34:00+08:00      3.339801e-01      4.356079
16              node-2                       system.net.packets_out.count     455.414500           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      2.427800e+03      4.330968
17   checkoutservice-1  container.node-6.container_network_receive_MB....       0.062435           2 2022-03-21 23:58:00+08:00 2022-03-21 23:59:00+08:00      3.305759e-01      4.294723
18  checkoutservice2-0  mesh.source.checkoutservice2.jaeger-collector....       0.001625           2 2022-03-21 23:33:00+08:00 2022-03-21 23:34:00+08:00      8.250000e-03      4.076923
19              node-4                        system.net.udp.in_datagrams     357.309000           2 2022-03-21 23:56:00+08:00 2022-03-21 23:57:00+08:00      1.705550e+03      3.773319,                     cmdb_id  number_of_runs
0                adservice2              34
1               frontend2-0              21
2                 adservice              18
3                    node-2              15
4   recommendationservice-0              14
5  productcatalogservice2-0              14
6                    node-4              13
7   productcatalogservice-0              12
8                    node-3              12
9             cartservice-2              11)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required kernel variables exist
for var in ['metric_fault_runs', 'runs_per_cmdb_top10', 'trace_thresholds', 'trace_window_df', 'log_thresholds', 'log_window_df']:
    if var not in globals():
        raise RuntimeError(f"Required variable '{var}' not found in kernel.")

# 1) Build candidate list
# Top 5 distinct cmdb_id by maximum breach_ratio from metric_fault_runs
if isinstance(metric_fault_runs, pd.DataFrame) and not metric_fault_runs.empty:
    top_by_breach = (
        metric_fault_runs
        .dropna(subset=['breach_ratio'])
        .groupby('cmdb_id', as_index=False)['breach_ratio']
        .max()
        .sort_values('breach_ratio', ascending=False)
        .head(5)
    )['cmdb_id'].astype(str).tolist()
else:
    top_by_breach = []

# Top 5 by number_of_runs from runs_per_cmdb_top10
if isinstance(runs_per_cmdb_top10, pd.DataFrame) and not runs_per_cmdb_top10.empty:
    top_by_count = runs_per_cmdb_top10['cmdb_id'].astype(str).head(5).tolist()
else:
    top_by_count = []

# Union, preserve order: first breach list then count list, unique, limit 10
candidate_list = []
for c in top_by_breach + top_by_count:
    if c not in candidate_list:
        candidate_list.append(c)
    if len(candidate_list) >= 10:
        break

# 2) For each candidate compute aggregates
rows = []
for cm in candidate_list:
    # Metric aggregates
    if isinstance(metric_fault_runs, pd.DataFrame) and not metric_fault_runs.empty:
        m = metric_fault_runs[metric_fault_runs['cmdb_id'] == cm]
        if not m.empty:
            metric_max_breach_ratio = m['breach_ratio'].dropna().max()
            metric_number_of_runs = int(m.shape[0])
            metric_earliest_run_start = m['start_ts'].min()
        else:
            metric_max_breach_ratio = np.nan
            metric_number_of_runs = 0
            metric_earliest_run_start = pd.NaT
    else:
        metric_max_breach_ratio = np.nan
        metric_number_of_runs = 0
        metric_earliest_run_start = pd.NaT

    # Trace aggregates
    trace_anom_count = np.nan
    trace_first_above_ts = pd.NaT
    trace_first_below_ts = pd.NaT
    if isinstance(trace_window_df, pd.DataFrame) and not trace_window_df.empty and isinstance(trace_thresholds, pd.DataFrame) and not trace_thresholds.empty:
        tw = trace_window_df[trace_window_df['cmdb_id'] == cm].copy()
        if not tw.empty:
            twm = tw.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
            if twm.shape[0] > 0:
                # compute per-row conditions safely handling NaNs
                cond_above = twm['p95'].notna() & (twm['value'] > twm['p95'])
                cond_below = twm['p05'].notna() & (twm['value'] < twm['p05'])
                cond_any = cond_above | cond_below
                trace_anom_count = int(cond_any.sum())
                # earliest above/below
                if cond_above.any():
                    trace_first_above_ts = twm.loc[cond_above, 'ts'].min()
                if cond_below.any():
                    trace_first_below_ts = twm.loc[cond_below, 'ts'].min()
            else:
                trace_anom_count = 0
        else:
            trace_anom_count = 0
    else:
        trace_anom_count = np.nan

    # Log aggregates
    log_anom_count = np.nan
    log_first_above_ts = pd.NaT
    log_first_below_ts = pd.NaT
    if isinstance(log_window_df, pd.DataFrame) and not log_window_df.empty and isinstance(log_thresholds, pd.DataFrame) and not log_thresholds.empty:
        lw = log_window_df[log_window_df['cmdb_id'] == cm].copy()
        if not lw.empty:
            lwm = lw.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
            if lwm.shape[0] > 0:
                cond_above_l = lwm['p95'].notna() & (lwm['value'] > lwm['p95'])
                cond_below_l = lwm['p05'].notna() & (lwm['value'] < lwm['p05'])
                cond_any_l = cond_above_l | cond_below_l
                log_anom_count = int(cond_any_l.sum())
                if cond_above_l.any():
                    log_first_above_ts = lwm.loc[cond_above_l, 'ts'].min()
                if cond_below_l.any():
                    log_first_below_ts = lwm.loc[cond_below_l, 'ts'].min()
            else:
                log_anom_count = 0
        else:
            log_anom_count = 0
    else:
        log_anom_count = np.nan

    rows.append({
        'cmdb_id': cm,
        'metric_max_breach_ratio': metric_max_breach_ratio,
        'metric_number_of_runs': metric_number_of_runs,
        'metric_earliest_run_start': metric_earliest_run_start,
        'trace_anomaly_count': trace_anom_count,
        'trace_first_above_ts': trace_first_above_ts,
        'trace_first_below_ts': trace_first_below_ts,
        'log_anomaly_count': log_anom_count,
        'log_first_above_ts': log_first_above_ts,
        'log_first_below_ts': log_first_below_ts
    })

candidate_localization_table = pd.DataFrame(rows)

# Normalize types and sort by metric_max_breach_ratio desc (NaNs last)
candidate_localization_table['metric_max_breach_ratio'] = pd.to_numeric(candidate_localization_table['metric_max_breach_ratio'], errors='coerce')
candidate_localization_table = candidate_localization_table.sort_values('metric_max_breach_ratio', ascending=False).reset_index(drop=True)

# Keep output compact: limit to candidate_list length (<=10)
candidate_localization_table = candidate_localization_table.head(len(candidate_list))

# Display compact outputs: candidate list and aggregated table
candidate_list, candidate_localization_table

```
Out[4]:
```
summary = (
    "Candidate localization summary (incident window 2022-03-21 23:30–00:00 UTC+8):\n\n"
    "Candidates (union of top breach and top-run components): emailservice-2, cartservice-2, paymentservice-0, node-2, "
    "paymentservice-1, recommendationservice-0, adservice, adservice2, frontend2-0.\n\n"
    "Key findings (per aggregated table):\n"
    "- emailservice-2: highest metric breach_ratio (~21.36), 3 metric fault runs (earliest run start 23:41). Trace anomalies (10 points) and some log anomalies (2 points) occur (first trace above ~23:38, first trace below ~23:41).\n"
    "- cartservice-2: very large metric breaches (breach_ratio ~19), 11 fault runs (earliest 23:41). Trace anomalies present (15 points) and several log anomalies (4 points).\n"
    "- paymentservice-0 and paymentservice-1: both show large metric breaches (~19 and ~6.5) with multiple runs and accompanying trace anomalies (several points) and a few log anomalies.\n"
    "- node-2: many metric runs (15) with notable network byte/packet spikes (metric_max_breach_ratio ~8.4) but no matching trace/log anomalies in the window (trace/log counts show 0) — suggests node-level network/system metric spikes.\n"
    "- recommendationservice-0 and frontend2-0: have many trace anomalies (recommendationservice-0: 43 trace anomaly points; frontend2-0: 92 trace anomaly points). Recommendationservice-0 also has metric runs; frontend2-0 has low metric breach but heavy trace anomalies.\n"
    "- adservice and adservice2: many metric runs but small metric breach_ratios and no trace/log anomalies in the window.\n\n"
    "Interpretation and next steps:\n"
    "1) Primary leads: emailservice-2, cartservice-2, paymentservice-0/1, recommendationservice-0, and frontend2-0 — these show both metric breaches and trace/log anomalies and should be investigated first.\n"
    "2) node-2 is a distinct signal: multiple metric fault runs (network/packet/bytes spikes) without trace/log anomalies — investigate node-level networking or host resource issues.\n"
    "3) For each top candidate, inspect traces and logs around the listed earliest anomaly timestamps (UTC+8) to find root cause (network spikes, container resource contention, or service-level errors).\n\n"
    "This table (candidate_localization_table) is stored in kernel for follow-up drill-downs."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(['emailservice-2', 'cartservice-2', 'paymentservice-0', 'node-2', 'paymentservice-1', 'adservice2', 'frontend2-0', 'adservice', 'recommendationservice-0'],                    cmdb_id  metric_max_breach_ratio  metric_number_of_runs metric_earliest_run_start  trace_anomaly_count      trace_first_above_ts      trace_first_below_ts  log_anomaly_count        log_first_above_ts        log_first_below_ts
0           emailservice-2                21.355440                      3 2022-03-21 23:41:00+08:00                   10 2022-03-21 23:38:00+08:00 2022-03-21 23:41:00+08:00                  2 2022-03-21 23:33:00+08:00 2022-03-21 23:45:00+08:00
1            cartservice-2                19.000000                     11 2022-03-21 23:41:00+08:00                   15 2022-03-21 23:35:00+08:00 2022-03-21 23:38:00+08:00                  4 2022-03-21 23:44:00+08:00 2022-03-21 23:38:00+08:00
2         paymentservice-0                19.000000                      6 2022-03-21 23:36:00+08:00                   12 2022-03-21 23:47:00+08:00 2022-03-21 23:31:00+08:00                  1 2022-03-21 23:31:00+08:00                       NaT
3                   node-2                 8.409046                     15 2022-03-21 23:30:00+08:00                    0                       NaT                       NaT                  0                       NaT                       NaT
4         paymentservice-1                 6.496261                      5 2022-03-21 23:33:00+08:00                   12 2022-03-21 23:39:00+08:00 2022-03-21 23:32:00+08:00                  2 2022-03-21 23:54:00+08:00 2022-03-21 23:42:00+08:00
5  recommendationservice-0                 3.357143                     14 2022-03-21 23:49:00+08:00                   43 2022-03-21 23:32:00+08:00 2022-03-21 23:32:00+08:00                  3 2022-03-21 23:41:00+08:00 2022-03-21 23:38:00+08:00
6                adservice                 0.316398                     18 2022-03-21 23:54:00+08:00                    0                       NaT                       NaT                  0                       NaT                       NaT
7               adservice2                 0.146242                     34 2022-03-21 23:30:00+08:00                    0                       NaT                       NaT                  0                       NaT                       NaT
8              frontend2-0                 0.102514                     21 2022-03-21 23:36:00+08:00                   92 2022-03-21 23:30:00+08:00 2022-03-21 23:30:00+08:00                  0                       NaT                       NaT)```
```

In [ ]:
# In[5]:

import re
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) determine candidate list from kernel or fallback
default_candidates = ['emailservice-2','cartservice-2','paymentservice-0','paymentservice-1','node-2','recommendationservice-0','frontend2-0','adservice','adservice2']
candidates = candidate_list if ('candidate_list' in globals() and isinstance(candidate_list, (list,tuple)) and len(candidate_list)>0) else default_candidates
# limit to max 10 unique
candidates = list(dict.fromkeys(candidates))[:10]

# Validate required trace data
if ('trace_window_df' not in globals()) or ('trace_thresholds' not in globals()):
    candidate_trace_anomalies = pd.DataFrame()  # store empty
    candidate_trace_anomalies_top50 = pd.DataFrame({'info': ['<EMPTY> missing trace_window_df or trace_thresholds']})
else:
    tw = trace_window_df[trace_window_df['cmdb_id'].isin(candidates)].copy()
    th = trace_thresholds.copy()
    if tw.empty:
        candidate_trace_anomalies = pd.DataFrame()
        candidate_trace_anomalies_top50 = pd.DataFrame({'info': ['<EMPTY> no trace rows for candidates in window']})
    else:
        # merge thresholds (global) into window rows
        twm = tw.merge(th, on=['cmdb_id','trace_name'], how='left')
        # safe boolean masks
        twm['cond_above'] = twm['p95'].notna() & (twm['value'] > twm['p95'])
        twm['cond_below'] = twm['p05'].notna() & (twm['value'] < twm['p05'])
        # group and aggregate per (cmdb_id, trace_name)
        def agg_fn(g):
            count_total = int(g.shape[0])
            count_above = int(g['cond_above'].sum())
            count_below = int(g['cond_below'].sum())
            earliest_above = g.loc[g['cond_above'], 'ts'].min() if g['cond_above'].any() else pd.NaT
            earliest_below = g.loc[g['cond_below'], 'ts'].min() if g['cond_below'].any() else pd.NaT
            maxv = g['value'].max()
            minv = g['value'].min()
            return pd.Series({
                'count_total': count_total,
                'count_above_p95': count_above,
                'count_below_p05': count_below,
                'earliest_above_ts': earliest_above,
                'earliest_below_ts': earliest_below,
                'max_value_in_window': maxv,
                'min_value_in_window': minv,
                'p95': g['p95'].iloc[0] if 'p95' in g.columns else np.nan,
                'p05': g['p05'].iloc[0] if 'p05' in g.columns else np.nan
            })
        grouped = (
            twm
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .apply(lambda g: agg_fn(g))
            .reset_index()
        )

        if grouped.empty:
            candidate_trace_anomalies = pd.DataFrame()
            candidate_trace_anomalies_top50 = pd.DataFrame({'info': ['<EMPTY> no aggregated trace anomalies for candidates']})
        else:
            # parse direction and peer_component from trace_name
            def parse_trace_name(tn):
                if pd.isna(tn):
                    return ('unknown', np.nan)
                # check self first
                if 'trace.self' in tn:
                    return ('self', np.nan)
                m = re.search(r'trace\.(from|to)_([A-Za-z0-9\-]+)', tn)
                if m:
                    direction = 'from' if m.group(1) == 'from' else 'to'
                    peer = m.group(2)
                    return (direction, peer)
                # fallback unknown
                return ('unknown', np.nan)

            parsed = grouped['trace_name'].apply(parse_trace_name)
            grouped['direction'] = parsed.map(lambda x: x[0])
            grouped['peer_component'] = parsed.map(lambda x: x[1])

            # compute first anomaly timestamp (earliest of above/below), leaving NaT if none
            grouped['first_anom_ts'] = grouped[['earliest_above_ts','earliest_below_ts']].min(axis=1)

            # sort by earliest anomaly ascending (NaT -> last)
            grouped_sorted = grouped.sort_values('first_anom_ts', na_position='last').reset_index(drop=True)

            # reorder columns as requested
            candidate_trace_anomalies = grouped_sorted[[
                'cmdb_id','trace_name','direction','peer_component','p95','p05',
                'count_total','count_above_p95','count_below_p05',
                'earliest_above_ts','earliest_below_ts','max_value_in_window','min_value_in_window','first_anom_ts'
            ]].copy()

            # store full result and prepare compact top50
            candidate_trace_anomalies_top50 = candidate_trace_anomalies.head(50)

# Return compact outputs: candidate list and the aggregated trace anomalies (top 50)
candidates, candidate_trace_anomalies_top50

```
Out[5]:
```
Summary of trace anomaly scan for candidates (using trace thresholds, incident window 2022-03-21 23:30–00:00 UTC+8):

- Candidate list examined: emailservice-2, cartservice-2, paymentservice-0, node-2, paymentservice-1, adservice2, frontend2-0, adservice, recommendationservice-0.

- Results: multiple trace series for these candidates show anomalies (values above global P95 or below P05). The script produced an aggregated table (stored as candidate_trace_anomalies) and the top 50 rows (sorted by earliest anomaly timestamp) were returned.

Key findings (from the top results):
- frontend2-0 shows the earliest and many anomalies starting at 23:30 — e.g., trace.to_shippingservice2-0.duration_p95, trace.self.duration_mean, trace.to_shippingservice2-0.row_count and trace.to_adservice2-0.row_count all show both above‑P95 and below‑P05 events. Peer components implicated: shippingservice2-0, adservice2-0, root.
- Several cartservice-2 and paymentservice-1 / paymentservice-0 trace series show anomalies clustered around ~23:35–23:41 (e.g., traces from frontend-2, from_checkoutservice-1). These traces have both above‑P95 and below‑P05 occurrences.
- recommendationservice-0 and other services also appear later in the list with multiple anomalous trace series.
- node-2 (a candidate from metrics) had no trace anomalies in the window — indicating node-level metric spikes may not have corresponding trace-level latency/anomaly signals.
- Many trace series show both directions (from/to) and both high and low deviations, suggesting cross-service interaction variability rather than a single isolated latency spike.

Actionable next steps:
1. Prioritize investigating frontend2-0 → shippingservice2-0 and frontend2-0 → adservice2-0 interactions around 23:30 (earliest anomalies).
2. Inspect traces and request-level logs for cartservice-2 and paymentservice-0/1 around their earliest anomaly timestamps (~23:35–23:41).
3. For node-2, focus on node-level metrics (network/packets/bytes spikes) because trace data shows no anomaly — that points to host/network issues rather than application-level latency.
4. Use the stored candidate_trace_anomalies table for detailed drill-down per trace_name and the exact earliest_above_ts/earliest_below_ts for each series.

The original code execution output of IPython Kernel is also provided below for reference:

(['emailservice-2', 'cartservice-2', 'paymentservice-0', 'node-2', 'paymentservice-1', 'adservice2', 'frontend2-0', 'adservice', 'recommendationservice-0'],              cmdb_id                                  trace_name direction      peer_component        p95        p05  count_total  count_above_p95  count_below_p05         earliest_above_ts         earliest_below_ts  max_value_in_window  min_value_in_window             first_anom_ts
0        frontend2-0    trace.to_shippingservice2-0.duration_p95        to  shippingservice2-0   0.000236   0.000076           30                2                2 2022-03-21 23:30:00+08:00 2022-03-21 23:45:00+08:00             0.000280             0.000071 2022-03-21 23:30:00+08:00
1        frontend2-0                    trace.self.duration_mean      self                 NaN   0.005090   0.004666           30                2                2 2022-03-21 23:30:00+08:00 2022-03-21 23:31:00+08:00             0.005217             0.004589 2022-03-21 23:30:00+08:00
2        frontend2-0       trace.to_shippingservice2-0.row_count        to  shippingservice2-0  25.550000   4.450000           30                2                2 2022-03-21 23:31:00+08:00 2022-03-21 23:30:00+08:00            27.000000             4.000000 2022-03-21 23:30:00+08:00
3        frontend2-0             trace.to_adservice2-0.row_count        to        adservice2-0  62.550000  37.450000           30                2                2 2022-03-21 23:30:00+08:00 2022-03-21 23:31:00+08:00            63.000000            28.000000 2022-03-21 23:30:00+08:00
4        frontend2-0               trace.from_root.duration_mean      from                root   0.052561   0.046368           30                2                2 2022-03-21 23:30:00+08:00 2022-03-21 23:31:00+08:00             0.053637             0.045893 2022-03-21 23:30:00+08:00
..               ...                                         ...       ...                 ...        ...        ...          ...              ...              ...                       ...                       ...                  ...                  ...                       ...
45     cartservice-2                    trace.self.duration_mean      self                 NaN   0.000059   0.000000           30                1                0 2022-03-21 23:41:00+08:00                       NaT             0.000455             0.000000 2022-03-21 23:41:00+08:00
46  paymentservice-1  trace.from_checkoutservice-1.duration_mean      from   checkoutservice-1   0.000173   0.000148            6                1                1 2022-03-21 23:41:00+08:00 2022-03-21 23:46:00+08:00             0.000173             0.000148 2022-03-21 23:41:00+08:00
47  paymentservice-1   trace.from_checkoutservice-1.duration_p95      from   checkoutservice-1   0.000173   0.000148            6                1                1 2022-03-21 23:41:00+08:00 2022-03-21 23:46:00+08:00             0.000173             0.000148 2022-03-21 23:41:00+08:00
48     cartservice-2          trace.from_frontend-2.duration_p95      from          frontend-2   0.001000   0.000000           30                1                0 2022-03-21 23:41:00+08:00                       NaT             0.007500             0.000000 2022-03-21 23:41:00+08:00
49     cartservice-2         trace.from_frontend-2.duration_mean      from          frontend-2   0.000312   0.000000           30                2                0 2022-03-21 23:41:00+08:00                       NaT             0.001500             0.000000 2022-03-21 23:41:00+08:00

[50 rows x 14 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate list specified by user
candidates_log = ['frontend2-0','shippingservice2-0','adservice2-0','cartservice-2',
                  'paymentservice-0','paymentservice-1','emailservice-2','node-2']

# Validate presence of required kernel variables
for var in ['log_thresholds', 'log_window_df', 'error_logs_df']:
    if var not in globals():
        raise RuntimeError(f"Required variable '{var}' not found in kernel.")

# Filter windowed logs to candidates
lw = log_window_df[log_window_df['cmdb_id'].isin(candidates_log)].copy()

if lw.empty:
    # Empty indicator table
    candidate_log_anomalies = pd.DataFrame({'info': ['<EMPTY> no log rows for candidates in the incident window']})
else:
    # Merge global thresholds (precomputed) into window rows
    lmerged = lw.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

    # Safe conditions for anomalies
    lmerged['cond_above'] = lmerged['p95'].notna() & (lmerged['value'] > lmerged['p95'])
    lmerged['cond_below'] = lmerged['p05'].notna() & (lmerged['value'] < lmerged['p05'])

    # Aggregation per (cmdb_id, log_name)
    def agg_logs(g):
        count_total = int(g.shape[0])
        count_above = int(g['cond_above'].sum())
        count_below = int(g['cond_below'].sum())
        earliest_above = g.loc[g['cond_above'], 'ts'].min() if g['cond_above'].any() else pd.NaT
        earliest_below = g.loc[g['cond_below'], 'ts'].min() if g['cond_below'].any() else pd.NaT
        maxv = g['value'].max()
        minv = g['value'].min()
        return pd.Series({
            'p95': g['p95'].iloc[0] if 'p95' in g.columns else np.nan,
            'p05': g['p05'].iloc[0] if 'p05' in g.columns else np.nan,
            'count_total': count_total,
            'count_above_p95': count_above,
            'count_below_p05': count_below,
            'earliest_above_ts': earliest_above,
            'earliest_below_ts': earliest_below,
            'max_value_in_window': maxv,
            'min_value_in_window': minv
        })

    agg = (
        lmerged
        .groupby(['cmdb_id','log_name'], as_index=False)
        .apply(lambda g: agg_logs(g))
        .reset_index()
    )

    if agg.empty:
        candidate_log_anomalies = pd.DataFrame({'info': ['<EMPTY> no aggregated log anomalies for candidates']})
    else:
        # compute earliest anomaly timestamp for sorting
        agg['first_anom_ts'] = agg[['earliest_above_ts','earliest_below_ts']].min(axis=1)

        # Order columns as requested
        candidate_log_anomalies = agg[[
            'cmdb_id','log_name','p95','p05','count_total','count_above_p95','count_below_p05',
            'earliest_above_ts','earliest_below_ts','max_value_in_window','min_value_in_window','first_anom_ts'
        ]].copy()

        # Sort by earliest non-null anomaly timestamp ascending, NaT last
        candidate_log_anomalies = candidate_log_anomalies.sort_values('first_anom_ts', na_position='last').reset_index(drop=True)

        # Keep full table in kernel variable; compact display later

# Search error_logs_df for candidate cmdb_ids and return up to 20 rows (timestamp, cmdb_id, message)
if 'error_logs_df' in globals():
    el = error_logs_df[error_logs_df['cmdb_id'].isin(candidates_log)].copy()
    if el.empty:
        candidate_error_logs_subset = pd.DataFrame({'info': ['<EMPTY> no error_logs for candidates']})
    else:
        # Select up to 20 rows and only requested columns; ensure ts is present
        cols = ['timestamp','cmdb_id','message']
        available_cols = [c for c in cols if c in el.columns]
        candidate_error_logs_subset = el[available_cols].head(20).reset_index(drop=True)
else:
    candidate_error_logs_subset = pd.DataFrame({'info': ['<EMPTY> error_logs_df not present in kernel']})

# Final compact outputs (top 20 rows of aggregated log anomalies and up to 20 error log rows)
# Display variable names (per kernel rules)
candidate_log_anomalies.head(20), candidate_error_logs_subset

```
Out[6]:
```
Summary of log-anomaly scan for specified candidates (window 2022-03-21 23:30–00:00 UTC+8):

- Cartservice-2:
  - log.row_count: 30 samples, 2 points above P95 and 2 below P05. Earliest below‑P05 at 23:38, earliest above‑P95 at 23:44. Values ranged 123 → 228 (P95=203.05, P05=135.95). This is the strongest log throughput fluctuation among candidates.
  - log.error_count: all zeros in-window (no error_count anomalies).

- Paymentservice-0:
  - log.row_count: 19 samples, 1 point above P95 (earliest 23:31), no below‑P05. Values ranged 3 → 11 (P95=6.50, P05=3.00).
  - log.error_count: all zeros.

- Paymentservice-1:
  - log.row_count: 19 samples, 1 above P95 (earliest 23:54) and 1 below P05 (earliest 23:42). Values ranged 1 → 9 (P95=6.30, P05=1.90).
  - log.error_count: all zeros.

- Emailservice-2:
  - log.row_count: 16 samples, 1 above P95 (earliest 23:33) and 1 below P05 (earliest 23:45). Values ranged 1 → 9 (P95=8.25, P05=1.75).
  - log.error_count: all zeros.

- Other candidates (frontend2-0, shippingservice2-0, adservice2-0, node-2):
  - No matching log rows in the incident window (or no anomalies for the specified log_name) — indicated as NA/empty in the aggregated results.

- Error logs:
  - No error_log entries found for these candidates in the loaded error_logs.csv (empty result).

Actionable notes:
- Prioritize cartservice-2 for log throughput investigation (earliest anomaly 23:38, both high and low deviations).
- Investigate paymentservice-0 (early anomaly 23:31) and emailservice-2 (23:33) next, then paymentservice-1 (23:42).
- No explicit error messages were found in error_logs.csv for these components; focus on traffic/throughput and cross-service traces around the listed earliest timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id         log_name     p95     p05  count_total  count_above_p95  count_below_p05         earliest_above_ts         earliest_below_ts  max_value_in_window  min_value_in_window             first_anom_ts
0  paymentservice-0    log.row_count    6.50    3.00           19                1                0 2022-03-21 23:31:00+08:00                       NaT                 11.0                  3.0 2022-03-21 23:31:00+08:00
1    emailservice-2    log.row_count    8.25    1.75           16                1                1 2022-03-21 23:33:00+08:00 2022-03-21 23:45:00+08:00                  9.0                  1.0 2022-03-21 23:33:00+08:00
2     cartservice-2    log.row_count  203.05  135.95           30                2                2 2022-03-21 23:44:00+08:00 2022-03-21 23:38:00+08:00                228.0                123.0 2022-03-21 23:38:00+08:00
3  paymentservice-1    log.row_count    6.30    1.90           19                1                1 2022-03-21 23:54:00+08:00 2022-03-21 23:42:00+08:00                  9.0                  1.0 2022-03-21 23:42:00+08:00
4     cartservice-2  log.error_count    0.00    0.00           30                0                0                       NaT                       NaT                  0.0                  0.0                       NaT
5    emailservice-2  log.error_count    0.00    0.00           16                0                0                       NaT                       NaT                  0.0                  0.0                       NaT
6  paymentservice-0  log.error_count    0.00    0.00           19                0                0                       NaT                       NaT                  0.0                  0.0                       NaT
7  paymentservice-1  log.error_count    0.00    0.00           19                0                0                       NaT                       NaT                  0.0                  0.0                       NaT,                                    info
0  <EMPTY> no error_logs for candidates)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables:
# metric_fault_runs, trace_window_df, trace_thresholds, log_window_df, log_thresholds

target = 'cartservice-2'

# 1) metric fault runs for cartservice-2
if 'metric_fault_runs' in globals() and isinstance(metric_fault_runs, pd.DataFrame) and not metric_fault_runs.empty:
    _m_runs = metric_fault_runs[metric_fault_runs['cmdb_id'] == target].copy()
    if _m_runs.empty:
        cartservice_metric_runs_detail = pd.DataFrame({'info': [f"<EMPTY> no metric fault runs for {target}"]})
    else:
        # sort by start_ts ascending and keep all columns
        _m_runs = _m_runs.sort_values('start_ts', ascending=True).reset_index(drop=True)
        cartservice_metric_runs_detail = _m_runs.copy()
else:
    cartservice_metric_runs_detail = pd.DataFrame({'info': [f"<EMPTY> metric_fault_runs not available or empty"]})

# Prepare a compact view (limit 50) for display
if isinstance(cartservice_metric_runs_detail, pd.DataFrame) and 'info' in cartservice_metric_runs_detail.columns:
    cartservice_metric_runs_detail_view = cartservice_metric_runs_detail
else:
    cartservice_metric_runs_detail_view = cartservice_metric_runs_detail.head(50)

# 2) trace anomalies for cartservice-2 (trace_name contains 'from_frontend' or 'self')
if ('trace_window_df' not in globals()) or ('trace_thresholds' not in globals()):
    cartservice_trace_anomalies_detail = pd.DataFrame({'info': [f"<EMPTY> trace data or thresholds missing"]})
    cartservice_trace_anomalies_detail_view = cartservice_trace_anomalies_detail
else:
    tw = trace_window_df[trace_window_df['cmdb_id'] == target].copy()
    if tw.empty:
        cartservice_trace_anomalies_detail = pd.DataFrame({'info': [f"<EMPTY> no trace rows for {target} in window"]})
        cartservice_trace_anomalies_detail_view = cartservice_trace_anomalies_detail
    else:
        # merge global thresholds
        twm = tw.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
        # filter trace_name substrings
        mask = twm['trace_name'].str.contains('from_frontend', na=False) | twm['trace_name'].str.contains('self', na=False)
        twm_f = twm[mask].copy()
        if twm_f.empty:
            cartservice_trace_anomalies_detail = pd.DataFrame({'info': [f"<EMPTY> no matching trace_name patterns for {target}"]})
            cartservice_trace_anomalies_detail_view = cartservice_trace_anomalies_detail
        else:
            # compute booleans
            twm_f['is_above_p95'] = twm_f['p95'].notna() & (twm_f['value'] > twm_f['p95'])
            twm_f['is_below_p05'] = twm_f['p05'].notna() & (twm_f['value'] < twm_f['p05'])
            # select requested columns and sort by timestamp ascending
            cols = ['timestamp','trace_name','value','p95','p05','is_above_p95','is_below_p05']
            # ensure columns exist
            available = [c for c in cols if c in twm_f.columns]
            cartservice_trace_anomalies_detail = twm_f[available].sort_values('timestamp', ascending=True).reset_index(drop=True)
            cartservice_trace_anomalies_detail_view = cartservice_trace_anomalies_detail.head(50)

# 3) logs for cartservice-2
if ('log_window_df' not in globals()) or ('log_thresholds' not in globals()):
    cartservice_log_anomalies_detail = pd.DataFrame({'info': [f"<EMPTY> log data or thresholds missing"]})
    cartservice_log_anomalies_detail_view = cartservice_log_anomalies_detail
else:
    lw = log_window_df[log_window_df['cmdb_id'] == target].copy()
    if lw.empty:
        cartservice_log_anomalies_detail = pd.DataFrame({'info': [f"<EMPTY> no log rows for {target} in window"]})
        cartservice_log_anomalies_detail_view = cartservice_log_anomalies_detail
    else:
        lwm = lw.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
        lwm['is_above_p95'] = lwm['p95'].notna() & (lwm['value'] > lwm['p95'])
        lwm['is_below_p05'] = lwm['p05'].notna() & (lwm['value'] < lwm['p05'])
        cols_log = ['timestamp','log_name','value','p95','p05','is_above_p95','is_below_p05']
        available_log = [c for c in cols_log if c in lwm.columns]
        cartservice_log_anomalies_detail = lwm[available_log].sort_values('timestamp', ascending=True).reset_index(drop=True)
        cartservice_log_anomalies_detail_view = cartservice_log_anomalies_detail.head(50)

# Return compact views for confirmation (full tables stored in variables per requirement)
cartservice_metric_runs_detail_view, cartservice_trace_anomalies_detail_view, cartservice_log_anomalies_detail_view

```
Out[7]:
```
Summary for cartservice-2 (incident window 2022-03-21 23:30–00:00 UTC+8):

1) Metric fault runs (consecutive value > global P95, run_length ≥2)
- Multiple short anomalous runs were observed for cartservice-2 (see stored table cartservice_metric_runs_detail).
- Notable runs (examples):
  - 23:41–23:42 — mesh.destination.frontend.cartservice.* (run_length 2), breach_ratio ≈ 0.26.
  - 23:42–23:43 — container network receive (MB) spike, breach_ratio ≈ 5.04 (strong network receive spike).
  - 23:42–23:43 — increased network transmit/packet metrics (moderate breaches).
  - 23:50–23:51 — mesh.source cartservice -> redis-cart (tcp) spike (modest breach).
  - 23:55–23:56 — container CPU cfs throttling metrics (seconds and percent) with very large breach_ratio ≈ 19 (indicative of CPU throttling).
  - 23:57–23:59 — short runs for container FS usage and memory metrics (small breaches).
- Interpretation: clustered network traffic spikes around ~23:42 and a distinct CPU throttling signal around ~23:55 on the node/container hosting cartservice-2.

2) Trace anomalies (filtered: trace.from_frontend*, trace.self)
- Detailed trace rows for cartservice-2 are returned in cartservice_trace_anomalies_detail.
- Aggregate/behavioral notes:
  - Several interactions from frontend (from_frontend-0 / from_frontend-2) and self traces were captured around 23:30–23:41.
  - The per-row sample mostly shows values within thresholds in the printed slice, but aggregated scans indicate there are trace anomaly counts for cartservice-2 (some trace series show anomalous points around ~23:41).
  - Recommended focus timestamps: around 23:38–23:41 for frontend→cartservice traces.

3) Logs (log.row_count / log.error_count)
- Detailed log rows for cartservice-2 are in cartservice_log_anomalies_detail.
- Key findings (from aggregated results):
  - log.row_count had 30 points in-window with 2 above P95 and 2 below P05.
  - Earliest below‑P05 (low throughput) at ~23:38; earliest above‑P95 (high throughput) at ~23:44 (UTC+8).
  - log.error_count was zero in-window (no error_count anomalies).
- Interpretation: throughput fluctuated (both dips and spikes) in the window, consistent with the trace/metric signals.

Overall conclusion / recommended next steps
- Cartservice-2 shows correlated signals: network traffic spikes (~23:42), throughput dips/spikes in logs (~23:38–23:44), and a separate CPU throttling event (~23:55). These indicate transient network and resource contention issues on the host/container.
- Immediate investigation:
  1. Inspect node/container CPU throttling (cgroups) and host CPU metrics around 23:55 for the pod hosting cartservice-2.
  2. Inspect network interfaces and Istio/mesh metrics around 23:41–23:43 for spikes in bytes/packets (both TX and RX).
  3. Drill into traces for frontend→cartservice interactions around 23:38–23:44 and correlate with the log.row_count anomalies.
- All detailed rows and full tables are stored in kernel variables: cartservice_metric_runs_detail, cartservice_trace_anomalies_detail, cartservice_log_anomalies_detail for follow-up drill-down.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name  run_id  run_length                  start_ts                    end_ts  max_value_in_run           p95  breach_ratio
0   cartservice-2  mesh.destination.frontend.cartservice.istio_re...       1           2 2022-03-21 23:41:00+08:00 2022-03-21 23:42:00+08:00      2.452500e-02  1.941875e-02  2.629546e-01
1   cartservice-2  container.node-6.container_network_receive_MB....       1           2 2022-03-21 23:42:00+08:00 2022-03-21 23:43:00+08:00      3.395877e-01  5.618832e-02  5.043742e+00
2   cartservice-2  container.node-6.container_network_transmit_pa...       1           2 2022-03-21 23:42:00+08:00 2022-03-21 23:43:00+08:00      3.640000e+02  3.067750e+02  1.865374e-01
3   cartservice-2  container.node-6.container_network_receive_pac...       1           2 2022-03-21 23:42:00+08:00 2022-03-21 23:43:00+08:00      4.920000e+02  4.392000e+02  1.202186e-01
4   cartservice-2  mesh.source.cartservice.redis-cart.istio_tcp_r...       1           2 2022-03-21 23:50:00+08:00 2022-03-21 23:51:00+08:00      1.729000e+03  1.666925e+03  3.723923e-02
5   cartservice-2  container.node-6.container_cpu_cfs_throttled_s...       1           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      1.740780e-02  8.703900e-04  1.900000e+01
6   cartservice-2  container.node-6.container_cpu_cfs_throttled_p...       1           2 2022-03-21 23:55:00+08:00 2022-03-21 23:56:00+08:00      5.000000e-01  2.500000e-02  1.900000e+01
7   cartservice-2   container.node-6.container_fs_usage_MB./dev/vda1       1           3 2022-03-21 23:57:00+08:00 2022-03-21 23:59:00+08:00      1.006641e+01  1.001611e+01  5.021206e-03
8   cartservice-2   container.node-6.container_memory_working_set_MB       1           3 2022-03-21 23:57:00+08:00 2022-03-21 23:59:00+08:00      1.490254e+02  1.488850e+02  9.432094e-04
9   cartservice-2         container.node-6.container_memory_usage_MB       1           3 2022-03-21 23:57:00+08:00 2022-03-21 23:59:00+08:00      1.550840e+02  1.549436e+02  9.063280e-04
10  cartservice-2               container.node-6.container_last_seen       1           3 2022-03-21 23:57:00+08:00 2022-03-21 23:59:00+08:00      1.647878e+09  1.647878e+09  1.074109e-07,      timestamp                           trace_name    value        p95    p05  is_above_p95  is_below_p05
0   1647876600  trace.from_frontend-0.duration_mean   0.0000   0.000404   0.00         False         False
1   1647876600   trace.from_frontend-0.duration_p95   0.0000   0.001000   0.00         False         False
2   1647876600     trace.from_frontend-0.error_rate   0.0000   0.000000   0.00         False         False
3   1647876600      trace.from_frontend-0.row_count   8.0000   9.000000   6.00         False         False
4   1647876600  trace.from_frontend-2.duration_mean   0.0000   0.000312   0.00         False         False
..         ...                                  ...      ...        ...    ...           ...           ...
45  1647876780              trace.self.duration_p95   0.0002   0.000200   0.00         False         False
46  1647876780                trace.self.error_rate   0.0000   0.000000   0.00         False         False
47  1647876780                 trace.self.row_count  17.0000  22.000000  14.45         False         False
48  1647876840  trace.from_frontend-0.duration_mean   0.0000   0.000404   0.00         False         False
49  1647876840   trace.from_frontend-0.duration_p95   0.0000   0.001000   0.00         False         False

[50 rows x 7 columns],      timestamp         log_name  value     p95     p05  is_above_p95  is_below_p05
0   1647876600  log.error_count    0.0    0.00    0.00         False         False
1   1647876600    log.row_count  159.0  203.05  135.95         False         False
2   1647876660  log.error_count    0.0    0.00    0.00         False         False
3   1647876660    log.row_count  181.0  203.05  135.95         False         False
4   1647876720  log.error_count    0.0    0.00    0.00         False         False
..         ...              ...    ...     ...     ...           ...           ...
45  1647877920    log.row_count  159.0  203.05  135.95         False         False
46  1647877980  log.error_count    0.0    0.00    0.00         False         False
47  1647877980    log.row_count  169.0  203.05  135.95         False         False
48  1647878040  log.error_count    0.0    0.00    0.00         False         False
49  1647878040    log.row_count  151.0  203.05  135.95         False         False

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```